In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName('SupplyChainProject').getOrCreate()

In [ ]:
# Creating CSV Dataset
csv_data = """order_id,supplier_id,item_name,stock_level,delivery_date,status
101,5,Microchips Type-A,120,2026-06-15,Delivered
102,1,Aluminium Casings,5,null,Pending
103,2,Lithium Battery Packs,14,2026-06-28,Shipped
104,3,Copper Wiring Reels,200,2026-06-10,Delivered
105,4,Fiber Optic Transceivers,9,2026-06-01,Delivered
106,5,Microchips Type-B,45,2026-06-20,Delivered
107,2,Solid State Drives,3,2026-05-25,Delivered
108,1,Steel Brackets,88,null,Processing
109,3,Rubber Seals,350,2026-06-12,Delivered
110,4,Sensor Modules,12,2026-07-05,Shipped
111,5,Power Adapters,110,2026-06-18,Delivered
112,1,Cooling Fans,8,2026-06-05,Delivered
113,2,LED Displays,0,null,Pending
114,3,Capacitors Pack,500,2026-06-22,Delivered
115,4,Circuit Boards,19,2026-06-14,Delivered
"""

with open('supply_chain_data.csv', 'w') as f:
    f.write(csv_data)
print("CSV file created successfully!")

CSV file created successfully!


In [ ]:
# Loading Data using PySpark
df = spark.read.csv('supply_chain_data.csv', header=True, inferSchema=True, nullValue="null")
print("Initial Data:")
df.show()
print("Schema:")
df.printSchema()

Initial Data:
+--------+-----------+--------------------+-----------+-------------+----------+
|order_id|supplier_id|           item_name|stock_level|delivery_date|    status|
+--------+-----------+--------------------+-----------+-------------+----------+
|     101|          5|   Microchips Type-A|        120|   2026-06-15| Delivered|
|     102|          1|   Aluminium Casings|          5|         NULL|   Pending|
|     103|          2|Lithium Battery P...|         14|   2026-06-28|   Shipped|
|     104|          3| Copper Wiring Reels|        200|   2026-06-10| Delivered|
|     105|          4|Fiber Optic Trans...|          9|   2026-06-01| Delivered|
|     106|          5|   Microchips Type-B|         45|   2026-06-20| Delivered|
|     107|          2|  Solid State Drives|          3|   2026-05-25| Delivered|
|     108|          1|      Steel Brackets|         88|         NULL|Processing|
|     109|          3|        Rubber Seals|        350|   2026-06-12| Delivered|
|     110|    

In [ ]:
# Data Cleaning and Transformation
df = df.na.drop()
print("Data after dropping null values:")
df.show()

# Formatting date
df = df.withColumn("delivery_date", to_date(col("delivery_date"), "yyyy-MM-dd"))

current_timeline = lit("2026-06-16")
df = df.withColumn("delay_days", datediff(current_timeline, col("delivery_date")))
df = df.withColumn("is_delayed", when(col("delay_days") > 0, "Yes").otherwise("No"))
print("Data after formatting date and adding delay columns:")
df.show()


Data after dropping null values:
+--------+-----------+--------------------+-----------+-------------+---------+
|order_id|supplier_id|           item_name|stock_level|delivery_date|   status|
+--------+-----------+--------------------+-----------+-------------+---------+
|     101|          5|   Microchips Type-A|        120|   2026-06-15|Delivered|
|     103|          2|Lithium Battery P...|         14|   2026-06-28|  Shipped|
|     104|          3| Copper Wiring Reels|        200|   2026-06-10|Delivered|
|     105|          4|Fiber Optic Trans...|          9|   2026-06-01|Delivered|
|     106|          5|   Microchips Type-B|         45|   2026-06-20|Delivered|
|     107|          2|  Solid State Drives|          3|   2026-05-25|Delivered|
|     109|          3|        Rubber Seals|        350|   2026-06-12|Delivered|
|     110|          4|      Sensor Modules|         12|   2026-07-05|  Shipped|
|     111|          5|      Power Adapters|        110|   2026-06-18|Delivered|
|     1

In [ ]:
# Filtering delayed shipments
delayed_shipments = df.filter(col("is_delayed") == "Yes")
print("Delayed Shipments:")
delayed_shipments.show()

Delayed Shipments:
+--------+-----------+--------------------+-----------+-------------+---------+----------+----------+
|order_id|supplier_id|           item_name|stock_level|delivery_date|   status|delay_days|is_delayed|
+--------+-----------+--------------------+-----------+-------------+---------+----------+----------+
|     101|          5|   Microchips Type-A|        120|   2026-06-15|Delivered|         1|       Yes|
|     104|          3| Copper Wiring Reels|        200|   2026-06-10|Delivered|         6|       Yes|
|     105|          4|Fiber Optic Trans...|          9|   2026-06-01|Delivered|        15|       Yes|
|     107|          2|  Solid State Drives|          3|   2026-05-25|Delivered|        22|       Yes|
|     109|          3|        Rubber Seals|        350|   2026-06-12|Delivered|         4|       Yes|
|     112|          1|        Cooling Fans|          8|   2026-06-05|Delivered|        11|       Yes|
|     115|          4|      Circuit Boards|         19|   2026-

In [ ]:
# Group by supplier and count delayed orders
delayed_orders_by_supplier = delayed_shipments.groupBy("supplier_id").count()
print("Delayed Orders by Supplier:")
delayed_orders_by_supplier.show()

Delayed Orders by Supplier:
+-----------+-----+
|supplier_id|count|
+-----------+-----+
|          1|    1|
|          3|    2|
|          5|    1|
|          4|    2|
|          2|    1|
+-----------+-----+



In [ ]:
# Save processed data to CSV or Parquet
df.write.mode("overwrite").csv("processed_supply_chain_data", header=True)
df.write.mode("overwrite").parquet("processed_supply_chain_data_parquet")
print("Processed data saved successfully!")

Processed data saved successfully!
